VRP의 과정을 통해서, COST가 최저인 경로를 분석하였다.
COST는 기타 소요 시간(충전, 승하차 등), 수요, 그리고 거리 이 3가지 요소를 반영하였다.

다만, 한계점이 존재했다. 
1. outgoing.csv를 통해서 시간대 별 인기 노선을 발견하였으나, 시간대 별 노선이었으므로 1시간만에 17개의 지역을 돌기는 매우 어려웠다.
2. 또한, HUB에서만 충전하도록 설정을 하였는데 배터리가 충분하였음에도 무조건 HUB만 돌면 충전이 되게끔 설정이 되었음
3. distance cost와 time cost를 하나로 통일시키는 작업을 하지 못하였음
4. ortools는 노선에 대한 fleet size를 하기에는 부적절한 툴이다. -> 교수님께서 조언해주신 대로 node couple를 여러개로 해서 진행을 해보았으나, 생성된 node couple은 모두 독립적이므로 fleet size 측정에 큰 오차가 발생할 것이다.

대안
1. 이번에는 Time Window의 개념을 도입.
-> 시간 단위 별로 수요 파악을 계속 진행하게 되므로, 위의 1의 한계점을 해결할 수 있다.(시간별로 이동하는 결과를 보여줄 수 있으므로)
-> 또한, 지금 vehicle을 1개로 하여 전체적인 수요 노선을 파악하고 있지만 추후 vehicle의 개수를 2개 이상으로 진행하게 된다면 한 node에 대해서 vehicle이 2개가 되는 불상사가 발생할 수 있다.
따라서, 이러한 문제점을 탈피하기 위해서 Time Window 개념을 도입해 Separation을 보장해보려고 한다.
이때, ortools의 경우에는 노드를 1개 밖에 경유할 수 밖에 없는 문제점이 존재하므로 이 문제를 해결하기 위해 NAME 만 다르고 모든 게 같은 NODE 여러 개(1, 11, 111, ....)를 생성해, if문을 이용해 vehicle 들이 Node list 중 2 군데 이상 위치한다면, collision으로 나오게 한다. (아마, if 문과 list를 활용하면 될 것 같음. 그렇게 len(list) => 2를 하면 되지 않을까 싶음)
-> 또한, 방문한 지역에 대해서 중복된 Node는 재방문 금지하도록 설정

2. UAV에 배터리 용량을 대입
-> 구체적인 mah나 용량이 아닌 [0,100]으로 설정해서 대략 160Km == 100%로 가정 (배터리 용량을 Capacity라 칭함)
-> 따라서, if Capacity < 30 -> depot(HUB) 이외의 node들의 penalty는 1000000으로 설정해 반드시 현 상황에서는 HUB를 방문하도록 함.
-> 또한, 충전시간도 100%를 한다고 가정
-> 이때, joby aviation에 따르면 20mile을 비행하기 위해서는 12mins의 충전시간을 요구한다고 기사에 나옴.
따라서, 1%를 충전하기 위해서는 약 36초가 걸린다고 가정할 수 있으므로 이 요소를 반영해 100% 완충할 때까지 진행.
-> 당연히, 충전시간 대에는 이동이 불가.(또한, capacity가 1이라면 타 UAV 접근 불가. But, HUB의 Capacity를 얼마로 하냐에 따라 결과는 달라질 듯)

3. Fleet size는 다른 툴을 활용
-> ortools와는 호환성이 너무 안좋음

4. time과 distance에 대해서는 objectives value를 이용해 최대한 통일성 있게 진행. -> 생각해보니, 이 작업은 그냥 time matrix로 통일하는 것이 맞는 것 같음. 본 분석에서는 항속을 등속도로 가정을 하였기에 거리와 시간은 사실상 같은 의미이다.

추구하는 방향
1. 우선, 버스 노선과 같은 방식으로 진행해보려고 한다.
-> 택시와 같은 방식으로 진행을 하게 된다면, UAV 운항이 종료되었을 때 인기가 있는 지역들에 대해서 Capacity가 초과되는 우려사항이 존재한다
-> 만일, 이 방식을 채택하게 된다면 굳이 배터리 요소는 반영하지 않아도 됨. VRP 과정을 진행했을 때, 거의 어지간한 지역을 다 비행할 수 있는 것을 확인할 수 있었다. 따라서, 경영학 적인 접근을 한 번 해보면 어떨까(타당성 조사 등)
-> But, UAV가 모든 node를 경유하게 된다면 거리가 160Km를 초과하게 되어 반드시 1회 충전은 필연적이다. 따라서, 경유하는 node를 늘리게 된다면 충전은 필연이게 됨.
-> 따라서, Time window 개념을 도입해 시간당 얼마나 많은 capacity를 반영할 수 있을지 분석 -> 안정성 판단

본 계획의 목표
1. distance matrix는 제거하고 time matrix를 이용해서 Time window와 cost 계산을 진행한다.
1-1. outgoing.csv에 있는 total_count의 column을 이용해 cost 계산을 진행.
2. 1과 1-1에서 계산한 각 cost에 대한 요소를 가중치 등을 활용해 수치를 최대한 통일 시킨 후에 objectives로 결과를 관찰
3. depot은 하나로 진행. 따라서, 버스의 노선의 형태로 어떤 노선을 채택해야 유의미한 결과를 얻을 수 있을지 판단 얻을 수 있을지 판단
4. 3에서 생성한 경로들을 토대로 ortool의 초기 경로 설정 알고리즘과 adddisjunction 기능을 활용해 capacity를 보장 -> 추후에 Time Window도 반영
5. 배터리 요소도 반영. 위에서 설계한 방향은 distance를 기준으로 배터리 측정을 하였으나, distance matrix는 제거했기에. 320km/h를 나눠 time matrix로 변환
6. 배터리가 30%이 이하가 될 경우, 위의 서비스를 실행.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import csv
import math

from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp



CONFIG = {
    "num_vehicles": 1,
    "start_depot": "Depot_Kintex",
    "end_depot": "Depot_Kintex",
    "penalty": 8000,
    "output_folder": "Kintex",
    "output_prefix": "Depot_Kintex",
    "designated_time": "08:00 ~ 08:59",
    "make_animation": True,
    "max_available_time": 18000,
    "max_available_distance": 1000000
}

# ==================================================
# 1. 기본 데이터 모델 생성
# ==================================================
def create_data_model():
    BASE_DIR = Path.cwd()

    vertiport = pd.read_csv(
        BASE_DIR / "dataset" / "버티포트_일반_후보지.csv",
        encoding="cp949"
    )

    vertiport_hub = pd.read_csv(
        BASE_DIR / "dataset" / "핵심허브_경도_위도_명칭.csv",
        encoding="cp949"
    )

    distance_vertiport = pd.concat(
    [vertiport, vertiport_hub],
    ignore_index=True
    )

    distance_vertiport["id"] = range(len(distance_vertiport))

    # 핵심 허브, 허브 역시 depot 뿐만 아니라 일반 node로 사용하기 위해 중복 value를 추가. But, 구분하기 위해 앞에 Depot을 추가
    distance_vertiport.loc[14, "NAME"] = "Depot_Bundang_Townhall"
    distance_vertiport.loc[15, "NAME"] = "Depot_Gimpo_Airport"
    distance_vertiport.loc[16, "NAME"] = "Depot_Beom_Gye"
    distance_vertiport.loc[17, "NAME"] = "Depot_Kintex"
    distance_vertiport.loc[18, "NAME"] = "Depot_Gwang_Myeong"
    distance_vertiport.loc[19, "NAME"] = "Depot_Incheon_Airport"

    lats = distance_vertiport["y_latitude"].values
    lons = distance_vertiport["x_longtitude"].values

    node_to_name = dict(zip(distance_vertiport["id"], distance_vertiport["NAME"]))
    name_to_node = dict(zip(distance_vertiport["NAME"], distance_vertiport["id"]))

    n = len(distance_vertiport)
    time_matrix = np.zeros((n, n), dtype=np.int64)
    distance_matrix = np.zeros((n, n), dtype=np.int64)


    UAV_SPEED_KMH = 320
    UAV_SPEED_M_PER_MIN = UAV_SPEED_KMH * 1000 / 60  # 5333.33 m/min
    
    time_matrix[i, j] = math.ceil(
        haversine_distance(
            lats[i], lons[i],
            lats[j], lons[j]
        ) / UAV_SPEED_M_PER_MIN
    )
    
    for i in range(n):
        for j in range(n):
            distance_matrix[i, j] = haversine_distance(
                lats[i], lons[i],
                lats[j], lons[j]
            )



    data = {}
    data["time_matrix"] = time_matrix.tolist()
    data["distance_matrix"] = distance_matrix.tolist()
    data["num_vehicles"] = CONFIG["num_vehicles"]
    data["node_to_name"] = node_to_name
    data["name_to_node"] = name_to_node

    data["starts"] = [name_to_node[CONFIG["start_depot"]]]
    data["ends"] = [name_to_node[CONFIG["end_depot"]]]
    # 모든 노드에 기본 승하차 시간 120초 적용
    service_time = [120] * n

    # HUB 노드 14~19에는 충전/지상처리 시간 900초 적용
    for hub_node in range(14, 20):
        service_time[hub_node] = 900

    data["service_time"] = service_time

    return data


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    return int(R * c)


# ==================================================
# 2. 수요 데이터 처리 / 시간대의 data를 가중치로 변환하는 초기작업
# ==================================================
def process_incheon(df): # df = incheon
    df = df.copy()

    df["Normal_Count"] = pd.to_numeric( # 간혹 csv에서 숫자이나, str로 인식하는 경우 존재. 따라서, to_numeric으로 int로 변형 + str이 이 함수에 적용 받으면, NaN.
        df["Normal_Count"],
        errors="coerce"
    )

    df = df.dropna(subset=["Normal_Count", "time"]) # NaN이 존재하는 행은 제거

    df["weight"] = df["Normal_Count"] / df["Normal_Count"].max() # 가중치 부여 방식: 최대값 나누기 각 value 값
    df["node_id"] = "Incheon_Airport" 

    return df[["time", "node_id", "weight"]]


def process_vertiport(df): # df = vertiport
    df = df.copy()

    df["outgoing_count"] = pd.to_numeric(
        df["outgoing_count"],
        errors="coerce"
    )

    df = df.dropna(subset=["outgoing_count", "time"])

    hour = pd.to_datetime(
        df["time"],
        format="%H:%M:%S",
        errors="coerce"
    ).dt.hour

    df["time"] = (
        hour.astype("Int64").astype(str).str.zfill(2) # 가지고 있는 data의 형태가 05:00의 형태이므로 앞에 이 형식에 맞는 코딩을 진행.
        + ":00 ~ "
        + hour.astype("Int64").astype(str).str.zfill(2)
        + ":59"
    )

    df["weight"] = df["outgoing_count"] / df["outgoing_count"].max()

    # cost_callback의 node_name과 맞추기 위해 NAME 사용
    df["node_id"] = df["NAME"] 

    return df[["time", "node_id", "weight"]]


# 위의 process_incheon과 process_vertiport의 계산과정을 통합하기 위함.
def merge_demand(df1, df2):
    return pd.concat([df1, df2], ignore_index=True)

# process_incheon과 process_vertiport에서 시간대 별로 계산한 값들을 표로 표현.
def build_time_dict(df):
    result = {}

    for t in df["time"].unique():
        sub = df[df["time"] == t]
        result[t] = dict(zip(sub["node_id"], sub["weight"]))

    return result


# 위의 과정을 최종적으로 정리한 부분
def demand():
    BASE_DIR = Path.cwd()

    incheon = pd.read_csv( 
        BASE_DIR / "dataset" / "인천공항_시간별_이용객수(2026.03).csv",
        encoding="utf-8-sig"
    )

    vertiport = pd.read_csv(
        BASE_DIR / "dataset" / "버티포트_지역의_시간대_별_인구유출_data.csv",
        encoding="utf-8-sig"
    )

    # SKT 데이터에서 공항 cell 제거, SKT 데이터의 인천공항 data는 매우 부정확하고 분석에 부정적인 영향을 미칠 것 같아. 제외함.
    vertiport = vertiport[
        vertiport["outgoing_cell"] != "8830e08c35fffff"
    ]

    incheon_df = process_incheon(incheon)
    vertiport_df = process_vertiport(vertiport)

    merged_demand = merge_demand(incheon_df, vertiport_df)
    time_demand = build_time_dict(merged_demand)

    print("시간대 개수:", len(time_demand))
    print("시간대 목록:", list(time_demand.keys()))

    return time_demand

# 기존 항로 이동에 대한 요소에서 time_spent 요소까지 추가
def time_spent(from_node, to_node, data):
    travel_time = data["time_matrix"][from_node][to_node]
    service_time = data["service_time"][from_node]

    return travel_time + service_time

# Adddimension에 거리제한도 넣기 위해서 def distance도 추가(단, cost 반영에는 안함)
def Move_distance(from_node, to_node, data):
    Travel_distance = data["distance_matrix"][from_node][to_node]

    return Travel_distance
# ==================================================
# 3. 비용 함수 
# ==================================================
def make_cost_callback(data, manager, time_demand, current_time):
    def cost_callback(from_index, to_index):
        # id를 node로 변환하는 작업
        from_node = int(manager.IndexToNode(from_index)) 
        to_node = int(manager.IndexToNode(to_index))

        
        travel_time = data["time_matrix"][from_node][to_node]

        node_name = data["node_to_name"][to_node] # 도착지점 node를 출력(최종도착지점 X) + 도착지점의 이름과 node를 출력
        demand_weight = time_demand[current_time].get(node_name, 0) # 해당 시간대의 가중치 

        try:
            demand_weight = float(demand_weight)
        except:
            demand_weight = 0

        """해당 node는 각각 김포공항과 인천공항이다. 공항은 도심지와 비교적 거리가 떨어져 있다. 
        따라서, distance_cost를 측정할 때, 다른 depot node 대비 높은 cost가 출력될 것이다.
        하지만, 거리가 멀더라도 공항 특성상 수요가 분명 높은 지점이므로 의도적으로 cost를 낮추었다."""
        alpha = 2.0

        if to_node == 19: # 인천공항
            cost = travel_time * 0.1 / (1 + alpha * demand_weight) + travel_time# distance에 0.1을 곱하고 분모에 alpha를 추가했다. 하지만, 아직 정확한 수치를 대입하지 못하였으므로 조금 더 분석을 진행해서 alpha와 상수의 적정값을 찾을 예정

        elif to_node == 15: # 김포공항, 비교적 공항에 더 가까워서
            cost = travel_time * 0.3 / (1 + alpha * demand_weight) + travel_time

        else:
            cost = travel_time / (1 + alpha * demand_weight) + travel_time

        return int(cost)

    
    return cost_callback
# ==================================================
# 4. 결과 출력
# ==================================================
def solution_to_text(data, manager, routing, solution, current_time):
    lines = []

    lines.append(f"\n===== {current_time} 결과 =====")
    lines.append(f"Objective: {solution.ObjectiveValue()}") # 궁극적으로 전페 cost를 측정하는 수치이고 objectiveValue()를 통해 최적화를 진행.

    max_route_distance = 0

    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        route_distance = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            name = data["node_to_name"][node]

            plan_output += f" {name} -> "

            previous_index = index
            index = solution.Value(routing.NextVar(index))

            route_distance += routing.GetArcCostForVehicle(
                previous_index,
                index,
                vehicle_id
            )

        end_node = manager.IndexToNode(index)
        end_name = data["node_to_name"][end_node]

        plan_output += f"{end_name}\n"
        plan_output += f"Distance of the route: {route_distance}m\n"

        lines.append(plan_output)
        max_route_distance = max(max_route_distance, route_distance)

    lines.append(f"Maximum of the route distances: {max_route_distance}m")

    return "\n".join(lines)


def print_solution(data, manager, routing, solution, current_time):
    text = solution_to_text(data, manager, routing, solution, current_time)
    print(text)
    return text

# csv화 진행
def extract_route_records(data, manager, routing, solution, current_time):
    summary_records = []
    step_records = []

    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)

        route_nodes = []
        route_names = []
        total_real_distance = 0
        total_spent_time = 0
        total_cost = 0
        step = 0

        while not routing.IsEnd(index):
            from_node = manager.IndexToNode(index)
            from_name = data["node_to_name"][from_node]

            route_nodes.append(from_node)
            route_names.append(from_name)

            previous_index = index
            index = solution.Value(routing.NextVar(index))

            to_node = manager.IndexToNode(index)
            to_name = data["node_to_name"][to_node]

            real_distance = data["distance_matrix"][from_node][to_node]
            arc_cost = routing.GetArcCostForVehicle(
                previous_index,
                index,
                vehicle_id
            )

            real_time = data["time_matrix"][from_node][to_node]
            arc_cost = routing.GetArcCostForVehicle(
                previous_index,
                index,
                vehicle_id
            )

            total_real_distance += real_distance
            total_spent_time += real_time
            total_cost += arc_cost
            route_efficiency = total_cost / total_real_distance if total_real_distance != 0 else 0
        
            step_records.append({
                "time": current_time,
                "vehicle_id": vehicle_id,
                "step": step,
                "from_node": from_node,
                "from_name": from_name,
                "to_node": to_node,
                "to_name": to_name,
                "real_distance": real_distance,
                "cost": arc_cost,
                "efficiency": route_efficiency,
                "real_distance": real_distance,
                "total_real_distance": total_real_distance,
                "total_spent_time" : total_spent_time,
                "objectives" : solution.ObjectiveValue()
            })

            step += 1

        end_node = manager.IndexToNode(index)
        end_name = data["node_to_name"][end_node]

        route_nodes.append(end_node)
        route_names.append(end_name)

        distance_list = []
        distance_list.append(total_real_distance)

        summary_records.append({
            "time": current_time,
            "vehicle_id": vehicle_id,
            "start_node": route_nodes[0],
            "end_node": route_nodes[-1],
            "route_nodes": " -> ".join(map(str, route_nodes)),
            "route_names": " -> ".join(route_names),
            "real_distance": total_real_distance,
            "cost": total_cost,
            "objectives" : solution.ObjectiveValue(),
            "total_spent_time" : total_spent_time,
            "total_real_distance": total_real_distance
        })

    return summary_records, step_records

# ==================================================
# 5. 경로 추출 및 애니메이션
# ==================================================
def extract_routes(data, manager, routing, solution):
    routes = []

    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        route = [manager.IndexToNode(index)]

        while not routing.IsEnd(index):
            index = solution.Value(routing.NextVar(index))
            route.append(manager.IndexToNode(index))

        routes.append(route)

    return routes

def generate_vehicle_colors(num_vehicles):
    color_pool = [
        "tomato",
        "cornflowerblue",
        "mediumseagreen",
        "gold",
        "orchid",
        "turquoise",
        "lime",
        "cyan",
        "magenta",
        "orange",
        "deepskyblue",
        "springgreen",
        "violet",
        "hotpink",
        "khaki",
        "dodgerblue",
        "lightcoral",
        "aquamarine",
        "plum",
        "salmon"
    ]

    if num_vehicles <= len(color_pool):
        return random.sample(color_pool, num_vehicles)
    else:
        return random.choices(color_pool, k=num_vehicles)

def animate_routes(data, manager, routing, solution, current_time):
    routes = extract_routes(data, manager, routing, solution)
    print("routes =", routes)

    pos = {
    0: (0, 0),

    # 상단 클러스터
    1: (-2, 5), 2: (2, 5), 3: (-4, 4), 4: (-3, 4),
    5: (1, 3), 6: (3, 3), 7: (-1, 2), 8: (2, 2),

    # 중간
    9: (1, 0), 10: (4, 0),

    # 하단 클러스터
    11: (-3, -2), 12: (-2, -2), 13: (-1, -3),
    14: (2, -3), 15: (-4, -4), 16: (3, -4),

    # 기존 허브
    17: (0, 4),
    18: (5, 1),
    19: (-5, -1),

}
    

    colors = generate_vehicle_colors(data["num_vehicles"])

    segments = []

    for vehicle_id, route in enumerate(routes):
        if len(route) <= 2 and route[0] == route[-1]:
            continue

        for i in range(len(route) - 1):
            a = route[i]
            b = route[i + 1]
            segments.append((vehicle_id, a, b))

    print("segments =", segments)

    if len(segments) == 0:
        print("애니메이션 생성할 경로가 없습니다.")
        return None

    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("black")

    ax.set_xlim(-7, 7)
    ax.set_ylim(-7, 7)
    ax.set_xticks(range(-7, 7))
    ax.set_yticks(range(-7, 7))
    ax.grid(True, color="gray", alpha=0.5, linewidth=0.8)
    ax.set_aspect("equal")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for node, (x, y) in pos.items():
        if node in data["starts"]:
            face = "lightgray"
            edge = "black"
            txt = "black"
        else:
            face = "white"
            edge = "white"
            txt = "black"

        circle = Circle(
            (x, y),
            0.23,
            facecolor=face,
            edgecolor=edge,
            linewidth=2,
            zorder=5
        )

        ax.add_patch(circle)
        ax.text(
            x,
            y,
            str(node),
            ha="center",
            va="center",
            fontsize=12,
            color=txt,
            zorder=6
        )

    drawn_artists = []

    def update(frame):
        
        vehicle_id, a, b = segments[frame]
        color = colors[vehicle_id % len(colors)]

        x1, y1 = pos[a]
        x2, y2 = pos[b]

        line, = ax.plot(
            [x1, x2],
            [y1, y2],
            color=color,
            linewidth=4,
            solid_capstyle="round",
            zorder=2
        )

        arrow = ax.annotate(
            "",
            xy=(x1 * 0.45 + x2 * 0.55, y1 * 0.45 + y2 * 0.55),
            xytext=(x1 * 0.55 + x2 * 0.45, y1 * 0.55 + y2 * 0.45),
            arrowprops=dict(
                arrowstyle="-|>",
                color=color,
                lw=2.2,
                mutation_scale=22
            ),
            zorder=3
        )

        drawn_artists.append(line)
        drawn_artists.append(arrow)

        ax.set_title(
            f"{current_time} | Step {frame + 1}: vehicle {vehicle_id}, {a} -> {b}",
            color="white"
        )

        return drawn_artists

    anim = FuncAnimation(
        fig,
        update,
        frames=len(segments),
        interval=800,
        repeat=False,
        blit=False
    )

    plt.tight_layout()

    safe_time = (
        current_time
        .replace(":", "")
        .replace(" ", "")
        .replace("~", "_")
    )
    
    BASE_DIR = Path.cwd()

    save_path = BASE_DIR /"dataset"/CONFIG['output_folder']/f"{CONFIG['start_depot']}_vrp_animation_{safe_time}.gif"

    anim.save(save_path, writer="pillow", fps=1)
    print(f"GIF 저장 완료: {save_path}")

    plt.show()

    return anim


# ==================================================
# 6. 시간대별 VRP 실행
# ==================================================
def run_vrp_for_time(current_time, time_demand, make_animation=False, penalty = CONFIG["penalty"]):
    data = create_data_model()

    manager = pywrapcp.RoutingIndexManager(
        len(data["time_matrix"]),
        data["num_vehicles"],
        data["starts"],
        data["ends"]
    ) # node의 이동 참고자료

    routing = pywrapcp.RoutingModel(manager) # manager를 기반으로 routing을 진행

    cost_callback = make_cost_callback(
        data,
        manager,
        time_demand,
        current_time
    )

    transit_callback_index = routing.RegisterTransitCallback(cost_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]
    
    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        travel_time = data["time_matrix"][from_node][to_node]
        service_time = data["service_time"][from_node]

        return travel_time + service_time

    distance_callback_index = routing.RegisterTransitCallback(distance_callback)
    time_callback_index = routing.RegisterTransitCallback(time_callback)

    # 일부 노드는 방문하지 않아도 되게 허용


    optional_nodes = []

    all_nodes = set(range(len(data["time_matrix"])))
    depot_nodes = set(data["starts"] + data["ends"])

    customer_nodes = sorted(list(all_nodes - depot_nodes))

    for node in customer_nodes:
        routing.AddDisjunction(
            [manager.NodeToIndex(node)],
            penalty
        )
        optional_nodes.append(node)

    print("penalty =", penalty)
    print("optional node count =", len(optional_nodes))
    print("optional nodes =", optional_nodes)
    print("expected penalty if all skipped =", len(optional_nodes) * penalty)   

    dimension_name = "Distance"
    dimension_name_1 = "Time"

    routing.AddDimension(
        distance_callback_index,
        0,
        CONFIG["max_available_distance"], # m 단위임 / km안으로 비행을 완료하라.
        True,
        dimension_name
    )
    

    routing.AddDimension(
        time_callback_index,
        0,
        CONFIG["max_available_time"], # s 단위/ k 초 안에 전부 비행을 완료하라
        True,
        dimension_name_1
    )

    distance_dimension = routing.GetDimensionOrDie(dimension_name) # Distance라는 이름의 dimension을 가져와라
    #distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC # 가장 비용이 적은 간선을 골라 경로를 생성
    )
    search_parameters.time_limit.seconds = 40

    solution = routing.SolveWithParameters(search_parameters) # 지금까지 정의한 모든 조건을 가지고 최적 경로를 계산

    if solution:
        result_text = print_solution(
            data,
            manager,
            routing,
            solution,
            current_time
        )

        if make_animation:
            anim = animate_routes(
                data,
                manager,
                routing,
                solution,
                current_time
            )
            print(anim)

        return {
            "text": result_text,
            "data": data,
            "manager": manager,
            "routing": routing,
            "solution": solution
        }

    else:
        result_text = f"\n===== {current_time} 결과 없음 ====="
        print(result_text)

        return {
            "text": result_text,
            "data": None,
            "manager": None,
            "routing": None,
            "solution": None
        }


# ==================================================
# 7. 전체 실행
# ==================================================
def main():
    BASE_DIR = Path.cwd()
    time_demand = demand()

    all_summary_records = []
    all_step_records = []

    penalty = CONFIG["penalty"]

    for current_time in time_demand.keys():
        result = run_vrp_for_time(
            current_time,
            time_demand,
            make_animation=False,
            penalty=penalty
        )

        if result is None or result["solution"] is None:
            continue

        summary_records, step_records = extract_route_records(
            result["data"],
            result["manager"],
            result["routing"],
            result["solution"],
            current_time
        )

        all_summary_records.extend(summary_records)
        all_step_records.extend(step_records)

    summary_df = pd.DataFrame(all_summary_records)
    steps_df = pd.DataFrame(all_step_records)

    output_dir = BASE_DIR / "dataset" / CONFIG["output_folder"]
    output_dir.mkdir(parents=True, exist_ok=True)

    output_prefix = CONFIG["output_prefix"]

    summary_df.to_csv(
        output_dir / f"{output_prefix}_route_summary_{penalty}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    steps_df.to_csv(
        output_dir / f"{output_prefix}_route_steps_{penalty}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    run_vrp_for_time(
        CONFIG["designated_time"],
        time_demand,
        make_animation=CONFIG["make_animation"],
        penalty=penalty
    )

    print("CSV 저장 완료")
    print(f'{CONFIG["designated_time"]} 시간대의 애니메이션 저장 완료')



if __name__ == "__main__":
    main()